In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_core.documents import Document

doc1 = Document(
    page_content="This is the content of document 1",
    metadata={"source": "doc1.txt"}
) # this is how a document looks like

In [3]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("Map of AI.pdf")
documensts = loader.load()

d:\Coading\GenAI\ShiaPrasad Valluru Course\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
documensts[1]

Document(metadata={'producer': 'PyPDF', 'creator': 'Google', 'creationdate': '', 'title': 'Map of AI', 'source': 'Map of AI.pdf', 'total_pages': 66, 'page': 1, 'page_label': '2'}, page_content='A Teacher’s Perspective\n  \nWe are also in the same boat\nMy goal for 2025\nBarely covered 20 percent\nHopped playlist for playlist\nGot frustrated')

In [5]:
len(documensts)

66

In [6]:
from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=50, separator="\n")

docs = text_splitter.split_documents(documents=documensts)

In [7]:
len(docs)

70

In [8]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_store = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory="chroma_db",
    collection_name="map_of_ai"
)


C:\Users\Aditya\AppData\Local\Temp\ipykernel_6920\1531490168.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


In [9]:
vector_store = Chroma(persist_directory="chroma_db", embedding_function=embeddings, collection_name="map_of_ai")
# now we can use this vector store for retrieval

In [10]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3} # this will return top 3 similar documents
)
# it also takes mmr as search type for max marginal relevance search

retriever.invoke("What is AI?")

[Document(id='cae67fd4-2c7c-4ff9-87ed-0ed292f05941', metadata={'producer': 'PyPDF', 'title': 'Map of AI', 'page_label': '64', 'total_pages': 66, 'page': 63, 'creator': 'Google', 'source': 'Map of AI.pdf', 'creationdate': ''}, page_content='P\nE\nO\nP\nL\nE\n● Educators – teach others how to use \nAI effectively and responsibly.'),
 Document(id='7193346c-72b4-4eb6-83de-3aac4ce7706b', metadata={'creationdate': '', 'page_label': '64', 'total_pages': 66, 'title': 'Map of AI', 'page': 63, 'producer': 'PyPDF', 'creator': 'Google', 'source': 'Map of AI.pdf'}, page_content='P\nE\nO\nP\nL\nE\n● Educators – teach others how to use \nAI effectively and responsibly.'),
 Document(id='dff218a4-e787-4121-b7e8-fdb8ece48f18', metadata={'page': 63, 'creator': 'Google', 'title': 'Map of AI', 'source': 'Map of AI.pdf', 'page_label': '64', 'total_pages': 66, 'producer': 'PyPDF', 'creationdate': ''}, page_content='P\nE\nO\nP\nL\nE\n● Educators – teach others how to use \nAI effectively and responsibly.')]

In [11]:
from langchain_community.vectorstores import FAISS

vector_store_faiss = FAISS.from_documents(
    documents=docs,
    embedding=embeddings,
)

vector_store_faiss.save_local("faiss_db")
# this is how we can create and save FAISS vector store locally

In [12]:
vector_store_faiss = FAISS.load_local(
    "faiss_db",
    embeddings=embeddings,
    allow_dangerous_deserialization=True
)
# this is how we retrieve the FAISS vector store


# once this is retrieved, we can use it as a retriever similar to Chroma
retriever_faiss = vector_store_faiss.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

In [18]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

message = """
Answer this question using the provided context only.
If the information is not available in the context, just reply with "i dont know"

{input}

Context:

{context}
"""

prompt = ChatPromptTemplate.from_messages([("human", message)])

# LLM (Gemini)
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0.7, timeout=30)

# Simple RAG helper (avoids langchain.chains dependency)
def rag_answer(query: str, retriever, llm, prompt_template_message: str, k: int = 3):
    """Retrieve top-k documents, build a short context, format the prompt, and call the LLM."""
    # retriever in this notebook is a Runnable-like retriever; use invoke to get documents
    docs = retriever.invoke(query)
    if not docs:
        return "No documents found for query."
    # build context from top-k docs
    context = "\n\n".join([d.page_content for d in docs[:k]])
    # fill template
    filled = prompt_template_message.format(input=query, context=context)
    # call LLM; many wrappers return an AIMessage-like object with .content
    response = llm.invoke(filled)
    return getattr(response, "content", str(response))

# Example usage
print("Invoking RAG helper for query: 'tell me about reimbursement policies'\n")
answer = rag_answer("tell me about reimbursement policies", retriever, llm, message, k=3)
print(answer)


Invoking RAG helper for query: 'tell me about reimbursement policies'

i dont know
i dont know
